# 策略概述

**DRL 門檻選擇式 v4** 是交易期（$T = 126$ 交易日）的學習型交易引擎：
神經網路依形成期特徵，**為每組配對每期選擇一個交易門檻組合**（或選擇不交易），
選定後交由標準 Z-Score 狀態機執行整個交易期。

1. **動作選單（9 個）**：SKIP（不交易）+ 8 組 $(entry_z, exit_z)$ 門檻組合
2. **決策網路**：MLP 以 12 維形成期特徵預測 9 個動作的期望報酬，取 argmax
3. **反事實標籤**：每期對全部 9 個動作逐一模擬實際報酬，作為訓練標籤（全資訊監督回歸）
4. **Walk-forward 訓練**：每期決策只用「交易期已於本期開始前結束」的歷史樣本（無前視）
5. 訓練樣本不足時自動退回基準動作 $(2.0, 0.0)$

實作模組：`strategies/trading/drl_threshold_trading.py`（`ThresholdNet` + `Trading` 類）。
使用本引擎的策略：SSD Rolling DRL THR、Agglomerative Fundamentals DRL THR
（各自借用對應 Z-Score 策略的形成期配對）。


# 參考文獻與引用對應


## 文獻 1：Kim & Kim (2019)

> Kim, T., & Kim, H. Y. (2019). Optimizing the pairs-trading strategy using deep reinforcement learning with trading and stop-loss boundaries. *Complexity*, 2019, Article 3582516.

**參考部分**：

- 核心框架：讓學習器**輸出交易邊界（進場／出場門檻）而非逐日持倉動作**——
  agent 的動作空間是門檻組合的離散集合，實際交易由既定規則依門檻執行
- 以配對的歷史特徵作為狀態輸入，門檻選擇以期為單位（而非以日為單位）

**為何參考**：

- 本引擎的動作空間設計（8 組 $(entry_z, exit_z)$ + SKIP）直接採用此文獻的門檻選擇式框架：
  把決策粒度從「每日」壓縮到「每配對每期一次」，
  大幅降低動作空間維度，避免模型對日級價格噪音做擇時
- SKIP 動作是對其框架的自然延伸：允許學習器拒絕不值得交易的配對

📄 文獻檔案：`ref/2021-Optimizing the Pairs-Trading Strategy Using Deep Reinforcement Learning with Trading and Stop-Loss Boundaries.pdf`


## 文獻 2：Sutton & Barto (2018)

> Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.

**參考部分**：

- 序貫決策問題的 MDP 形式化 $(S, A, P, R, \gamma)$，以及**探索－利用權衡**的成立條件：
  只有在「未執行的動作觀察不到報酬」（部分回饋）時才需要探索

**為何參考**：

- 用於界定本問題的性質：歷史配對期上**全部 9 個動作的報酬都能精確反事實回算**
  （對交易期價格逐一模擬 9 組門檻），屬於**全資訊回饋**——
  依此理論框架，問題退化為監督回歸，無需探索機制、無需價值迭代，
  這是本引擎採用「反事實標籤 + MSE 回歸」而非 Q-learning 的理論依據

✅ **已收錄於 `ref/` 資料夾內。**


## 文獻 3：統計套利機器學習方法綜述 (2025)

> A survey of statistical arbitrage pair trading with machine learning, deep learning, and reinforcement learning methods (2025).

**參考部分**：

- RL 應用於配對交易的動作空間設計分類：逐日持倉決策 vs 門檻／邊界決策兩大路線，
  及其在樣本效率與過度交易風險上的取捨

**為何參考**：

- 提供門檻選擇式路線在文獻中的定位；本引擎的設計取捨（決策粒度、費用敏感性）與其歸納一致

📄 文獻檔案：`ref/2025-A survey of statistical arbitrage pair trading with machine learning, deep learning, and reinforcement learning methods.pdf`


# 各階段行為

引擎對每個交易期的每組配對依序執行：特徵萃取 → 訓練 → 決策 → 反事實標籤生成 → 正式模擬。


## 階段 1：Spread 與 Z-Score（與 Z-Score 狀態機同口徑）

價格先做同樣的異常清理（單日 |漲跌幅| > 50% 以前值遞補），
Z-Score 以形成期凍結參數計算（路徑 B，標準化空間）：

$$Z_t = \text{clip}\left(\frac{P'_{A,t} - \beta P'_{B,t} - \mu_\epsilon^{form}}{\max(\sigma_\epsilon^{form}, 10^{-8})},\ -10,\ 10\right), \qquad
P'_{i,t} = \frac{\ln P_{i,t} - \mu_i^{form}}{\sigma_i^{form}}$$

spread 定義與 Z-Score 基準完全相同——確保門檻選擇是唯一的差異來源。


## 階段 2：12 維形成期特徵萃取

對**形成期**（非交易期）的 Z 序列與對數價格計算特徵，全部截尾至約 $[-3, 3]$：

| # | 特徵 | 計算 | 捕捉的資訊 |
| :---: | :--- | :--- | :--- |
| 1 | 期末 $Z$ | $\text{clip}(z_{-1}/3)$ | 進場初始偏離方向 |
| 2 | 期末 $|Z|$ | $\text{clip}(|z_{-1}|/3)$ | 偏離幅度 |
| 3 | 零穿越頻率 | $\text{clip}(zc \times 10)$ | 回歸訊號密度 |
| 4 | 半衰期 | $\ln(HL)/3$（AR(1) 估計，截尾 $[1, 252]$） | 回歸速度 |
| 5 | 近期波動 regime | 近 21 日 $\sigma_z$ ／全期 $\sigma_z - 1$ | 波動狀態變化 |
| 6 | 近期 $Z$ 趨勢 | $(z_{-1} - \bar{z}_{21})/3$ | 短期方向 |
| 7 | 兩股相關係數 | $\text{corr}(\ln P_A, \ln P_B)$ | 共動強度 |
| 8 | 波動率比 | $\sigma_A/\sigma_B - 1$ | 兩腳對稱性 |
| 9 | 對沖比例偏移 | $\beta - 1$ | 曝險不對稱 |
| 10 | Spread 振幅 | $\sigma_\epsilon \times 5$ | 費用可行性（振幅需覆蓋摩擦成本） |
| 11 | $|Z|>2$ 佔比 | $\text{mean}(|z| > 2)$ | 訊號出現頻率 |
| 12 | 最大 $|Z|$ | $\max|z| / 5$ | 極端偏離歷史 |

形成期資料不足 60 日時無法建構特徵 → 該配對直接使用基準動作。


## 階段 3：Walk-Forward 訓練（無前視保證）

**樣本資格**：緩衝區中每筆樣本記錄其交易期結束日 $t_e$，本期（交易起始日 $trade\_start_k$）
只用已完結的樣本訓練：

$$\text{eligible} = \{(f, r, t_e) \in \text{buffer} : t_e < trade\_start_k\}$$

**訓練條件與程序**：

- 樣本數 $\ge$ `thr_min_train_samples`（= 200）且較上次訓練有新樣本時才重新訓練
- `ThresholdNet`：$\text{Linear}(12 \to 64) \to \text{ReLU} \to \text{Linear}(64 \to 64) \to \text{ReLU} \to \text{Linear}(64 \to 9)$
- `Adam(lr = 10^{-3})`、`MSELoss`、40 epochs、batch $\min(4096, n)$
- 權重初始化與 batch 洗牌**不固定隨機種子**：單次回測數值為隨機變數，
  以 `run_drl_variance.py` 多次重跑報告分布統計

**狀態隔離**：網路／緩衝區以 `variant_id`（策略名稱 + Top N + 停損 + MSR）為鍵隔離，
同一行程內依序執行的不同參數變體不共用任何訓練狀態。


## 階段 4：決策

$$a^* = \begin{cases}
\arg\max_a \ \text{net}(f)_a & \text{if } n_{eligible} \ge 200 \text{ 且特徵可用} \\
(2.0,\ 0.0) \ \text{（基準動作）} & \text{otherwise}
\end{cases}$$

動作選單：

$$\mathcal{A} = \{\text{SKIP}\} \cup \{(e, x) : e \in \{1.5, 2.0, 2.5, 3.0\},\ x \in \{0.0, 0.5\}\}, \qquad |\mathcal{A}| = 9$$

**結構保證**：

1. 選單含基準 $(2.0, 0.0)$ → 本引擎的策略空間**包含** Z-Score 狀態機
2. 訓練不足時退回基準 → 樣本累積初期行為等同 Z-Score 狀態機
3. SKIP 提供「拒絕交易」的選擇權——預期報酬為負的配對可整期空手


## 階段 5：反事實標籤生成

無論本期實際選了哪個動作，都對**全部 8 個非 SKIP 動作**在本期交易資料上逐一模擬
（`_fast_threshold_pnl`，與正式模擬同一套狀態機邏輯與費用會計），
記錄各動作的期末報酬率（%）作為 9 維標籤向量（SKIP 恆為 0）：

$$r_a = \frac{\text{PnL}_a}{C_{pair}} \times 100, \quad a \in \mathcal{A}$$

樣本 $(f, r, t_e)$ 存入緩衝區，供**之後**的期別訓練使用（本期不用，見階段 3 資格規則）。

**性質**：因所有動作的報酬皆可觀察，這是**全資訊監督回歸**而非部分回饋的 bandit ——
無探索－利用問題（依據：Sutton & Barto 2018 的回饋結構分類）。


## 階段 6：以選定門檻執行正式模擬

選定 $(e, x)$ 後，整個交易期執行標準 Z-Score 狀態機：

| 條件 | 動作 |
| :--- | :--- |
| $Z_t > e$（空手時） | 空 A、多 B（風險中性配置 $v_A = C/W$、$v_B = |\beta|C/W$） |
| $Z_t < -e$ | 多 A、空 B |
| 空頭且 $Z_t \le x$／多頭且 $Z_t \ge -x$ | 平倉（可再進場） |
| 期末仍持倉 | `PERIOD_END_EXIT` 強制結算 |
| 選擇 SKIP | 整期 `HOLD_CASH (SKIP)` |

- 費用會計與 Z-Score 狀態機相同：進出場各扣 $(\text{fee} + \text{slippage}) \times$ 兩腳名目金額
- 最後一日不開新倉（$i < T-1$ 才允許進場）
- 輸出交易日誌欄位與 Z-Score 狀態機完全一致（`ZScore`、`Position`、`Status`、`Daily_Delta` 等）


# 參數總表

| 參數 | 值 | 說明 |
| :--- | :---: | :--- |
| 動作選單 | SKIP + $\{1.5,2.0,2.5,3.0\} \times \{0.0,0.5\}$ | 共 9 個動作 |
| 基準動作 | $(2.0, 0.0)$ | 訓練不足時的預設 |
| `drl_hidden_size` | 64 | MLP 隱藏層寬度（兩層） |
| `drl_lr` | $10^{-3}$ | Adam 學習率 |
| `thr_train_epochs` | 40 | 每次增量訓練 epoch 數 |
| `thr_min_train_samples` | 200 | 啟用網路決策的最低樣本數 |
| 特徵維度 | 12 | 形成期特徵（見階段 2） |
| 隨機種子 | 不固定 | 變異數以 `run_drl_variance.py` 多次重跑評估 |
| `capital_per_pair` | 10,000 | 每配對獨立資金 |
| `fee_rate` + `slippage_rate` | 0.001 + 0.001 | 與 Z-Score 狀態機相同 |
